In [8]:
import os
from dotenv import load_dotenv

# LangChain Document Loaders & Splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings & Vector Store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# LLM (Groq)
from langchain_groq import ChatGroq

# Prompts, Parsers & Runnables for LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [9]:
load_dotenv()

True

In [ ]:
pdf_path = "../../ABC_Pvt_Ltd_Policy_Handbook.pdf" 
loader = PyPDFLoader(pdf_path)
docs = loader.load()

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)
print(f"Total document chunks created: {len(splits)}")

Total document chunks created: 81


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",

)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1802.53it/s]


In [ ]:
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [25]:
llm = ChatGroq(
    temperature=0, 
    model_name="openai/gpt-oss-20b"
)

In [ ]:
system_prompt = (
    "You are an assistant for HR policy question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Keep the answer concise, clear and professional."
    "\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

In [ ]:
output_parser = StrOutputParser()

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [32]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [33]:
# পলিসি থেকে আপনার প্রশ্ন
question = "What is the policy regarding remote work and sick leave?"

response = chain.invoke(question)

In [34]:
print("Question:", question)
print("\nAnswer:\n", response)

Question: What is the policy regarding remote work and sick leave?

Answer:
 **Remote Work**  
The provided policy excerpts do not contain any information about remote‑work arrangements. I’m not aware of a specific remote‑work policy from the material you shared.

**Sick Leave**  
- Sick Leave can be taken for the employee’s own illness or to care for an ill dependent child, spouse, or parent, provided supporting evidence is supplied.  
- If Sick Leave is exhausted, the employee may use Casual Leave (CL) or Privilege Leave (PL); any further absence is treated as Loss of Pay.  
- The policy notes that sick leave exceeding 5 days is handled differently, but the exact procedure is not fully detailed in the excerpt.
